## RAG


In [6]:
import chromadb
from pypdf import PdfReader
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os


In [7]:
# The .env lives at the repo root, the database and index next to this script
ROOT = Path.cwd().parent


In [11]:
load_dotenv(ROOT / ".env.example")
API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"


In [12]:
print(API_KEY)

None


In [ ]:
client = OpenAI()

# 1. Read PDF
reader = PdfReader("../rag_docs/irs_pub_523.pdf")


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:

text = ""

for page in reader.pages:
    text += page.extract_text() or ""

# 2. Split text into chunks
chunk_size = 500

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

# 3. Create ChromaDB
chroma = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma.get_or_create_collection(
    name="pdf_documents"
)

# 4. Generate embeddings
for i, chunk in enumerate(chunks):

    embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk
    ).data[0].embedding

    collection.upsert(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embedding]
    )

print("PDF stored in ChromaDB!")